# LeetCode 694 — Number of Distinct Islands bằng Python

Link đề: https://leetcode.com/problems/number-of-distinct-islands/ · Độ khó: **Medium** · Chủ đề: Depth-First Search, Breadth-First Search, Union Find, Array, Hash Table, Matrix, Hash Function

> **Bài Premium (🔒).** Đề trong notebook được viết theo hiểu biết về bài; hãy đối chiếu lại với đề gốc nếu bạn có tài khoản Premium.

Skeleton LeetCode:

```python
class Solution:
    def numDistinctIslands(self, grid: List[List[int]]) -> int:
        ...
```

Ghi chú: bài chỉ dùng list, tuple, set và chuỗi của Python. Mọi đáp án trong notebook được đối chiếu với một oracle độc lập (thử mọi phép tịnh tiến giữa từng cặp đảo).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trên lưới nhị phân, một **đảo** là nhóm các ô `1` nối với nhau theo **4 hướng** (trên, dưới, trái, phải). Hai đảo được coi là **giống nhau** nếu **tịnh tiến** (dịch chuyển, không xoay, không lật) được đảo này trùng đảo kia. Đếm số đảo **khác nhau**.
2. Nắm các công cụ Python: duyệt lưới 2 chiều `grid[i][j]`, mảng hướng `(-1, 0, 1, 0, -1)`, kiểm tra biên, DFS/BFS (flood fill), `set` của các giá trị **hashable** (tuple, frozenset, chuỗi), `str.join`, sao chép sâu lưới, và Union-Find.
3. Đi từ **so sánh từng cặp đảo** → **chuẩn hóa tọa độ về gốc (tịnh tiến về `(0, 0)`) rồi bỏ vào set** → **chuỗi đường đi DFS có ký hiệu quay lui** → **cắt khung bao (bounding box) thành "ảnh"** → **Union-Find rồi chuẩn hóa từng nhóm**.
4. Hiểu mấu chốt: cần một **"dấu vân tay" (canonical form)** của mỗi đảo mà **không phụ thuộc vị trí** nhưng **phân biệt mọi hình dạng khác nhau**; hai đảo giống nhau ⇔ dấu vân tay bằng nhau.
5. Tránh các lỗi kinh điển: chỉ ghi hướng đi mà **quên ký hiệu quay lui** (hai hình khác nhau cho cùng chuỗi), quên chuẩn hóa tọa độ, dùng `list` làm phần tử của `set` (không hashable), coi đảo xoay/lật là giống nhau, và sửa lưới gốc rồi dùng lại.

## 2. Đề bài & ví dụ

Cho ma trận nhị phân `grid` kích thước `m x n`. **Đảo** là một nhóm ô `1` (đất) liên thông theo **4 hướng** (ngang hoặc dọc). Có thể giả sử bốn mép của lưới đều là nước.

Hai đảo được xem là **giống nhau** khi và chỉ khi có thể **tịnh tiến** (không xoay, không đối xứng gương) đảo này để trùng khớp đảo kia.

Trả về **số lượng đảo khác nhau**.

**Ví dụ 1**

```
1 1 0 0 0
1 1 0 0 0
0 0 0 1 1
0 0 0 1 1
```

- Output: `1`
- Giải thích: có hai đảo hình vuông `2 x 2`, chúng giống nhau (chỉ khác vị trí).

**Ví dụ 2**

```
1 1 0 1 1
1 0 0 0 0
0 0 0 0 1
1 1 0 1 1
```

- Output: `3`
- Giải thích: có 4 đảo: hình chữ "L" ở góc trên trái, đoạn ngang 2 ô ở trên phải, hình "┘" ở dưới phải, đoạn ngang 2 ô ở dưới trái. Hai đoạn ngang giống nhau, nên có 3 loại.

**Ràng buộc:**

- `m == grid.length`, `n == grid[i].length`
- `1 <= m, n <= 50`
- `grid[i][j]` là `0` hoặc `1`.

Chú ý: ô chỉ liên thông theo **4 hướng**, các ô chéo nhau **không** nối. Và **xoay/lật là khác nhau** (bài khó hơn, LeetCode 711, cho phép xoay/lật).

In [1]:
from typing import List, Optional
from collections import deque
import copy
import random
import sys
import time

sys.setrecursionlimit(20000)     # DFS đệ quy có thể sâu tới 50 * 50 = 2500 tầng

examples = [
    ([[1, 1, 0, 0, 0], [1, 1, 0, 0, 0], [0, 0, 0, 1, 1], [0, 0, 0, 1, 1]], 1),
    ([[1, 1, 0, 1, 1], [1, 0, 0, 0, 0], [0, 0, 0, 0, 1], [1, 1, 0, 1, 1]], 3),
]

def show(grid):
    for row in grid:
        print(" ".join("#" if x else "." for x in row))

for grid, expected in examples:
    show(grid)
    print("-> kỳ vọng", expected)
    print()

# # . . .
# # . . .
. . . # #
. . . # #
-> kỳ vọng 1

# # . # #
# . . . .
. . . . #
# # . # #
-> kỳ vọng 3



## 3. Kiến thức nền

### 3.1. Lưới 2 chiều: `grid[i][j]`, `m`, `n`

Lưới là **list của các list**: `grid[i]` là hàng `i`, `grid[i][j]` là ô ở hàng `i`, cột `j`. Số hàng `m = len(grid)`, số cột `n = len(grid[0])`. Hàng `0` ở trên cùng; cột `0` ở ngoài cùng bên trái.

In [2]:
grid = [[1, 1, 0],
        [0, 1, 0],
        [0, 0, 1]]
m, n = len(grid), len(grid[0])
print("m =", m, "n =", n)
print("grid[0] =", grid[0])
print("grid[1][1] =", grid[1][1], "| grid[2][2] =", grid[2][2])
for i in range(m):
    for j in range(n):
        if grid[i][j]:
            print(f"ô đất tại ({i}, {j})")

m = 3 n = 3
grid[0] = [1, 1, 0]
grid[1][1] = 1 | grid[2][2] = 1
ô đất tại (0, 0)
ô đất tại (0, 1)
ô đất tại (1, 1)
ô đất tại (2, 2)


### 3.2. Bốn hướng và mẹo `dirs = (-1, 0, 1, 0, -1)`

Từ ô `(i, j)`, bốn ô kề theo 4 hướng là `(i-1, j)` trên, `(i, j+1)` phải, `(i+1, j)` dưới, `(i, j-1)` trái. Mẹo gọn: mảng `dirs = (-1, 0, 1, 0, -1)`; với `h = 1..4`, bước đi là `(dirs[h-1], dirs[h])` cho ra đúng 4 cặp `(-1,0), (0,1), (1,0), (0,-1)` theo thứ tự **trên, phải, dưới, trái**. Ta sẽ dùng `h` (1 đến 4) làm "mã hướng".

**Kiểm tra biên:** ô `(x, y)` nằm trong lưới khi `0 <= x < m` và `0 <= y < n`. Luôn kiểm tra biên **trước** khi đọc `grid[x][y]`.

In [3]:
dirs = (-1, 0, 1, 0, -1)
names = {1: "trên", 2: "phải", 3: "dưới", 4: "trái"}
i, j = 1, 1
for h in range(1, 5):
    dx, dy = dirs[h - 1], dirs[h]
    x, y = i + dx, j + dy
    inside = 0 <= x < 3 and 0 <= y < 3
    print(f"h={h} ({names[h]:4s}) bước ({dx:2d},{dy:2d}) -> ({x},{y}) trong lưới 3x3: {inside}")

h=1 (trên) bước (-1, 0) -> (0,1) trong lưới 3x3: True
h=2 (phải) bước ( 0, 1) -> (1,2) trong lưới 3x3: True
h=3 (dưới) bước ( 1, 0) -> (2,1) trong lưới 3x3: True
h=4 (trái) bước ( 0,-1) -> (1,0) trong lưới 3x3: True


### 3.3. DFS (flood fill) tìm một đảo

**Flood fill** là "đổ loang": từ một ô đất, đi sang mọi ô đất kề, rồi từ các ô đó đi tiếp... cho đến khi hết. Để không đi lại ô cũ ta **đánh dấu đã thăm** (ở đây gán `grid[i][j] = 0`, biến đất thành nước). Hàm đệ quy dưới đây đếm số ô của đảo chứa ô bắt đầu.

In [4]:
def island_size(grid, i, j):
    m, n = len(grid), len(grid[0])
    grid[i][j] = 0                       # đánh dấu đã thăm
    total = 1
    for h in range(1, 5):
        x, y = i + dirs[h - 1], j + dirs[h]
        if 0 <= x < m and 0 <= y < n and grid[x][y]:
            total += island_size(grid, x, y)
    return total

g = [[1, 1, 0],
     [0, 1, 0],
     [0, 0, 1]]
print("đảo chứa (0,0) có", island_size(g, 0, 0), "ô")     # 3: (0,0),(0,1),(1,1)
print("sau khi thăm:", g)                                  # lưới gốc đã bị sửa!

đảo chứa (0,0) có 3 ô
sau khi thăm: [[0, 0, 0], [0, 0, 0], [0, 0, 1]]


Chú ý: hàm **sửa lưới** (đã gán `0`). Nếu cần dùng lưới gốc lần nữa, hãy sao chép trước bằng `copy.deepcopy(grid)` hoặc `[row[:] for row in grid]`.

### 3.4. Giá trị "hashable" và `set`

`set` chỉ chứa các giá trị **hashable** (có mã băm ổn định, bất biến): số, chuỗi, `tuple` (gồm các phần tử hashable), `frozenset`. **`list` và `set` thường không hashable** nên không bỏ được vào `set`. Để dùng làm "dấu vân tay" đưa vào set, ta đổi thành `tuple`, `frozenset` hoặc chuỗi.

In [5]:
s = set()
s.add((0, 1))                       # tuple: được
s.add(frozenset({(0, 0), (0, 1)}))  # frozenset: được
s.add("02-23-30")                   # chuỗi: được
print(len(s), s)
try:
    s.add([0, 1])                   # list: lỗi
except TypeError as e:
    print("TypeError:", e)
print((0, 1) == (0, 1), "-> hai tuple bằng nhau là một phần tử trong set")

3 {(0, 1), frozenset({(0, 1), (0, 0)}), '02-23-30'}
TypeError: unhashable type: 'list'
True -> hai tuple bằng nhau là một phần tử trong set


### 3.5. Tịnh tiến = trừ đi một mốc

**Tịnh tiến** một hình là cộng cùng một vector vào mọi ô. Hai hình trùng nhau sau tịnh tiến ⇔ nếu ta **trừ đi một mốc phù hợp** (một ô được chọn theo quy tắc **không phụ thuộc vị trí**, ví dụ ô nhỏ nhất theo thứ tự hàng rồi cột) thì hai tập tọa độ tương đối bằng nhau.

Ví dụ: đảo `{(0,3),(0,4)}` và `{(3,0),(3,1)}`. Trừ ô đầu: cả hai thành `{(0,0),(0,1)}`.

In [6]:
def normalize(cells):
    """Tịnh tiến về gốc: trừ đi hàng nhỏ nhất và cột nhỏ nhất."""
    r0 = min(r for r, c in cells)
    c0 = min(c for r, c in cells)
    return frozenset((r - r0, c - c0) for r, c in cells)

a = [(0, 3), (0, 4)]
b = [(3, 0), (3, 1)]
c = [(3, 0), (4, 0)]                # đoạn dọc: khác (không xoay)
print(normalize(a) == normalize(b))   # True
print(normalize(a) == normalize(c))   # False
print(sorted(normalize(a)))

True
False
[(0, 0), (0, 1)]


### 3.6. `str.join`

`"".join(list_chuỗi)` nối các chuỗi lại thành một chuỗi, ví dụ `"".join(["0", "2", "-2"]) = "02-2"`. Ta dùng nó để biến một danh sách các bước đi thành **một chuỗi** làm dấu vân tay.

In [7]:
steps = ["0", "2", "-2", "3", "-3", "0"]
print("".join(steps))
print(",".join(steps))

02-23-30
0,2,-2,3,-3,0


### 3.7. Union-Find (Disjoint Set Union) trong 5 dòng

**Union-Find** quản lý các nhóm rời nhau: `find(x)` trả về "đại diện" của nhóm chứa `x`; `union(a, b)` gộp hai nhóm. Ta dùng mảng `parent`, ban đầu `parent[x] = x`. `find` đi theo `parent` cho tới gốc (và nén đường đi). Cách này hợp với bài tìm "thành phần liên thông" như các đảo.

In [8]:
parent = list(range(6))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]   # nén đường đi
        x = parent[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

union(0, 1); union(1, 2); union(4, 5)
groups = {}
for x in range(6):
    groups.setdefault(find(x), []).append(x)
print(list(groups.values()))     # [[0, 1, 2], [3], [4, 5]]

[[0, 1, 2], [3], [4, 5]]


## 4. Ý tưởng nền tảng / trực giác

Tính tay với ví dụ 2:

```
    c0 c1 c2 c3 c4
r0   1  1  0  1  1
r1   1  0  0  0  0
r2   0  0  0  0  1
r3   1  1  0  1  1
```

Bốn đảo (tọa độ `(hàng, cột)`):

- A: `(0,0), (0,1), (1,0)` — chữ "L" lật (một góc trên trái).
- B: `(0,3), (0,4)` — đoạn ngang.
- C: `(2,4), (3,3), (3,4)` — hình "┘".
- D: `(3,0), (3,1)` — đoạn ngang.

**Chuẩn hóa** (trừ hàng nhỏ nhất, cột nhỏ nhất của chính đảo):

- A: `{(0,0), (0,1), (1,0)}`
- B: `{(0,0), (0,1)}`
- C: min hàng 2, min cột 3 → `{(0,1), (1,0), (1,1)}`
- D: `{(0,0), (0,1)}`

B và D **trùng nhau**; A, C khác nhau và khác B/D. Có đúng **3** dạng khác nhau.

**Trực giác:** "tịnh tiến về gốc" xóa thông tin về **vị trí**, giữ lại **hình dạng**. Mỗi đảo cho một "dấu vân tay" (tập tọa độ tương đối); đếm số dấu vân tay khác nhau bằng `set`.

In [9]:
grid = examples[1][0]
m, n = len(grid), len(grid[0])
seen = [[False] * n for _ in range(m)]
islands = []
for si in range(m):
    for sj in range(n):
        if grid[si][sj] and not seen[si][sj]:
            stack, cells = [(si, sj)], []
            seen[si][sj] = True
            while stack:
                i, j = stack.pop()
                cells.append((i, j))
                for h in range(1, 5):
                    x, y = i + dirs[h - 1], j + dirs[h]
                    if 0 <= x < m and 0 <= y < n and grid[x][y] and not seen[x][y]:
                        seen[x][y] = True
                        stack.append((x, y))
            islands.append(sorted(cells))

for isl in islands:
    print(isl, "->", sorted(normalize(isl)))
print("số đảo:", len(islands), "| số dạng khác nhau:", len({normalize(i) for i in islands}))

[(0, 0), (0, 1), (1, 0)] -> [(0, 0), (0, 1), (1, 0)]
[(0, 3), (0, 4)] -> [(0, 0), (0, 1)]
[(2, 4), (3, 3), (3, 4)] -> [(0, 1), (1, 0), (1, 1)]
[(3, 0), (3, 1)] -> [(0, 0), (0, 1)]
số đảo: 4 | số dạng khác nhau: 3


## 5. Cách 1: Brute Force (so sánh từng cặp đảo)

**Ý tưởng:** (1) tìm **mọi đảo** (mỗi đảo là danh sách ô, sắp theo thứ tự hàng rồi cột); (2) duyệt các đảo, giữ danh sách "đại diện" các dạng đã thấy; với mỗi đảo mới, **so sánh với từng đại diện**: nếu có đại diện giống nhau (qua tịnh tiến) thì bỏ qua, ngược lại thêm vào đại diện.

Hai đảo (đã sắp xếp theo thứ tự hàng rồi cột) giống nhau ⇔ cùng số ô và **hiệu tọa độ** giữa ô thứ `t` và ô đầu tiên **bằng nhau** với mọi `t`. (Ô đầu tiên theo thứ tự hàng rồi cột là mốc không phụ thuộc vị trí.)

In [10]:
def find_islands(grid):
    """Trả về danh sách các đảo; mỗi đảo là list ô (hàng, cột) đã sắp xếp. Không sửa lưới."""
    m, n = len(grid), len(grid[0])
    seen = [[False] * n for _ in range(m)]
    result = []
    for si in range(m):
        for sj in range(n):
            if grid[si][sj] and not seen[si][sj]:
                queue = deque([(si, sj)])
                seen[si][sj] = True
                cells = []
                while queue:
                    i, j = queue.popleft()
                    cells.append((i, j))
                    for h in range(1, 5):
                        x, y = i + dirs[h - 1], j + dirs[h]
                        if 0 <= x < m and 0 <= y < n and grid[x][y] and not seen[x][y]:
                            seen[x][y] = True
                            queue.append((x, y))
                result.append(sorted(cells))
    return result

def same_shape(a, b):
    if len(a) != len(b):
        return False
    (ar, ac), (br, bc) = a[0], b[0]
    return all((x - ar, y - ac) == (u - br, v - bc) for (x, y), (u, v) in zip(a, b))

def brute_force(grid):
    reps = []
    for isl in find_islands(grid):
        if not any(same_shape(isl, r) for r in reps):
            reps.append(isl)
    return len(reps)

for grid, expected in examples:
    got = brute_force(grid)
    print("->", got, "| kỳ vọng", expected)
    assert got == expected

-> 1 | kỳ vọng 1
-> 3 | kỳ vọng 3


**Độ phức tạp:** gọi `I` là số đảo, `S` là tổng số ô đất (`S <= m * n`). Tìm đảo `O(m * n)`. So sánh: với mỗi đảo so với tối đa `I` đại diện, mỗi lần `O(kích thước đảo)`; tệ nhất khoảng `O(I * S)`. Tổng **Time `O(m * n + I * S)`**, Space `O(m * n)`.

**Nút thắt:** số đảo `I` có thể lên tới `~ m * n / 2` (các ô lẻ loi), và tổng so sánh có thể lên đến hàng triệu phép. Cải tiến: đổi mỗi đảo thành một **dấu vân tay hashable** rồi dùng `set` (tra cứu `O(1)`), thay cho so sánh từng cặp.

## 6. Ý tưởng thuật toán / bước đột phá

Cần một hàm `fingerprint(đảo)` thỏa **hai tính chất**:

1. **Bất biến theo tịnh tiến:** dịch đảo đi đâu cũng cho cùng một kết quả.
2. **Phân biệt hình dạng:** hai đảo **khác hình** phải cho kết quả **khác nhau**.

Có nhiều cách xây `fingerprint`:

- **Tọa độ tương đối:** trừ mốc rồi dùng `frozenset`/`tuple` đã sắp xếp (Cách 2).
- **Đường đi DFS có ký hiệu quay lui:** ghi lại mã hướng mỗi lần đi vào ô mới **và** mỗi lần quay ra; cho một chuỗi (Cách 3).
- **Cắt khung bao:** cắt phần chữ nhật nhỏ nhất bao đảo, mã hóa thành "ảnh" (Cách 4).

Điểm tinh tế của đường đi DFS: nếu **chỉ** ghi hướng đi vào mà **không ghi quay lui**, hai hình khác nhau có thể sinh cùng chuỗi, ví dụ:

```
Hình P:  ##      Hình Q:  ##
         #.               .#
```

Từ ô đầu tiên của P: đi sang phải rồi (kết thúc, quay lại) đi xuống. Từ ô đầu tiên của Q: đi sang phải rồi đi xuống từ ô thứ hai. Cả hai đều chỉ ghi "phải, xuống", mặc dù hình khác nhau! Ghi thêm **ký hiệu quay lui** cho biết đoạn nào đã "rút lại" nên phân biệt được. Ta kiểm chứng bằng code:

In [11]:
def path_signature(cells, backtrack):
    """DFS trên tập ô `cells`, bắt đầu từ ô nhỏ nhất theo (hàng, cột). Trả về chuỗi đường đi."""
    cells = set(cells)
    start = min(cells)
    path, seen = [], set()
    def dfs(i, j, k):
        seen.add((i, j))
        path.append(str(k))
        for h in range(1, 5):
            x, y = i + dirs[h - 1], j + dirs[h]
            if (x, y) in cells and (x, y) not in seen:
                dfs(x, y, h)
        if backtrack:
            path.append(str(-k))
    dfs(*start, 0)
    return "".join(path)

P = [(0, 0), (0, 1), (1, 0)]
Q = [(0, 0), (0, 1), (1, 1)]
print("P và Q cùng hình? ", normalize(P) == normalize(Q), "(không)")
print("không quay lui  : P ->", path_signature(P, False), "| Q ->", path_signature(Q, False), "-> trùng nhau!")
print("có quay lui     : P ->", path_signature(P, True), "| Q ->", path_signature(Q, True))
assert path_signature(P, False) == path_signature(Q, False)
assert path_signature(P, True) != path_signature(Q, True)

P và Q cùng hình?  False (không)
không quay lui  : P -> 023 | Q -> 023 -> trùng nhau!
có quay lui     : P -> 02-23-30 | Q -> 023-3-20


## 7. Các cách tiếp theo

Tổng cộng **5 cách**: Cách 1 so sánh từng cặp (đã làm); Cách 2 BFS + tọa độ tương đối + `set`; Cách 3 DFS với chuỗi đường đi có quay lui (cách chính); Cách 4 cắt khung bao thành "ảnh"; Cách 5 Union-Find rồi chuẩn hóa từng nhóm.

### 7.1. Cách 2: BFS + chuẩn hóa tọa độ + `set`

**Giải thích từng đoạn:**

1. Tìm mọi đảo bằng BFS (như `find_islands`).
2. Với mỗi đảo, trừ đi **hàng nhỏ nhất** và **cột nhỏ nhất** của chính nó, thu được `frozenset` các ô tương đối.
3. Bỏ `frozenset` vào một `set`; kết quả là kích thước của `set`.

In [12]:
def normalized_set(grid):
    shapes = set()
    for isl in find_islands(grid):
        r0 = min(r for r, c in isl)
        c0 = min(c for r, c in isl)
        shapes.add(frozenset((r - r0, c - c0) for r, c in isl))
    return len(shapes)

for grid, expected in examples:
    got = normalized_set(grid)
    print("->", got, "| kỳ vọng", expected)
    assert got == expected

-> 1 | kỳ vọng 1
-> 3 | kỳ vọng 3


**Code nộp LeetCode (Cách 2):**

```python
class Solution:
    def numDistinctIslands(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        shapes = set()
        for si in range(m):
            for sj in range(n):
                if grid[si][sj] != 1:
                    continue
                grid[si][sj] = 0
                queue = deque([(si, sj)])
                cells = []
                while queue:
                    i, j = queue.popleft()
                    cells.append((i, j))
                    for dx, dy in ((-1, 0), (0, 1), (1, 0), (0, -1)):
                        x, y = i + dx, j + dy
                        if 0 <= x < m and 0 <= y < n and grid[x][y] == 1:
                            grid[x][y] = 0
                            queue.append((x, y))
                r0 = min(r for r, c in cells)
                c0 = min(c for r, c in cells)
                shapes.add(frozenset((r - r0, c - c0) for r, c in cells))
        return len(shapes)
```

**Độ phức tạp:** mỗi ô được thăm một lần; mỗi đảo tạo `frozenset` kích thước của nó. Time `O(m * n)` (băm `frozenset` tỉ lệ với kích thước đảo, tổng `O(S)`), Space `O(m * n)`.

### 7.2. Cách 3: DFS với chuỗi đường đi có ký hiệu quay lui (cách chính)

**Giải thích từng đoạn:**

1. Duyệt lưới; gặp ô đất chưa thăm thì bắt đầu một DFS mới và một chuỗi `path` rỗng.
2. `dfs(i, j, k)` (với `k` là mã hướng vừa đi vào ô này; ô bắt đầu dùng `k = 0`): đánh dấu thăm (gán `0`), thêm `k` vào `path`, rồi thử 4 hướng `h = 1..4`: nếu ô kề là đất thì `dfs(x, y, h)`.
3. **Sau khi thử xong cả 4 hướng**, thêm `-k` vào `path` (ký hiệu **quay lui**: "rút khỏi ô này").
4. Chuỗi `"".join(path)` là dấu vân tay; bỏ vào `set` chuỗi.

Thứ tự thử hướng cố định (trên, phải, dưới, trái) cho mọi đảo, nên hai đảo giống nhau (qua tịnh tiến) có cùng thứ tự duyệt và cùng chuỗi; hai đảo khác nhau cho chuỗi khác nhau (chuỗi mã hóa cây DFS một cách duy nhất).

In [13]:
def dfs_path(grid):
    m, n = len(grid), len(grid[0])
    shapes = set()
    path = []

    def dfs(i, j, k):
        grid[i][j] = 0
        path.append(str(k))
        for h in range(1, 5):
            x, y = i + dirs[h - 1], j + dirs[h]
            if 0 <= x < m and 0 <= y < n and grid[x][y]:
                dfs(x, y, h)
        path.append(str(-k))            # quay lui

    for i in range(m):
        for j in range(n):
            if grid[i][j]:
                dfs(i, j, 0)
                shapes.add("".join(path))
                path.clear()
    return len(shapes)

for grid, expected in examples:
    got = dfs_path(copy.deepcopy(grid))
    print("->", got, "| kỳ vọng", expected)
    assert got == expected

-> 1 | kỳ vọng 1
-> 3 | kỳ vọng 3


**Code nộp LeetCode (Cách 3):**

```python
class Solution:
    def numDistinctIslands(self, grid: List[List[int]]) -> int:
        def dfs(i: int, j: int, k: int):
            grid[i][j] = 0
            path.append(str(k))
            dirs = (-1, 0, 1, 0, -1)
            for h in range(1, 5):
                x, y = i + dirs[h - 1], j + dirs[h]
                if 0 <= x < m and 0 <= y < n and grid[x][y]:
                    dfs(x, y, h)
            path.append(str(-k))

        paths = set()
        path = []
        m, n = len(grid), len(grid[0])
        for i, row in enumerate(grid):
            for j, x in enumerate(row):
                if x:
                    dfs(i, j, 0)
                    paths.add("".join(path))
                    path.clear()
        return len(paths)
```

**Độ phức tạp:** mỗi ô được thăm một lần; chuỗi mỗi đảo dài `2 * kích thước đảo`. Time `O(m * n)`, Space `O(m * n)` (ngăn xếp đệ quy tối đa `m * n` tầng, cộng set chuỗi).

### 7.3. Cách 4: Cắt khung bao thành "ảnh"

**Ý tưởng khác:** thay vì liệt kê tọa độ, ta **vẽ lại đảo** trong khung chữ nhật nhỏ nhất bao nó (bounding box), dùng `#` cho ô thuộc **đảo này** và `.` cho mọi ô khác (kể cả đất của đảo khác nằm trong khung). Ảnh là một `tuple` các chuỗi hàng. Hai đảo giống nhau (tịnh tiến) có ảnh **y hệt**.

**Giải thích từng đoạn:**

1. Với mỗi đảo (danh sách ô), tính `r0, r1, c0, c1` là hàng/cột nhỏ nhất, lớn nhất.
2. Dựng ma trận `(r1 - r0 + 1) x (c1 - c0 + 1)` toàn `.`, rồi đặt `#` tại các ô của đảo (đã trừ `r0, c0`).
3. Biến mỗi hàng thành chuỗi, cả ảnh thành `tuple` để băm.

In [14]:
def bounding_box_picture(grid):
    pictures = set()
    for isl in find_islands(grid):
        r0 = min(r for r, c in isl); r1 = max(r for r, c in isl)
        c0 = min(c for r, c in isl); c1 = max(c for r, c in isl)
        canvas = [["."] * (c1 - c0 + 1) for _ in range(r1 - r0 + 1)]
        for r, c in isl:
            canvas[r - r0][c - c0] = "#"
        pictures.add(tuple("".join(row) for row in canvas))
    return len(pictures)

for grid, expected in examples:
    got = bounding_box_picture(grid)
    print("->", got, "| kỳ vọng", expected)
    assert got == expected

# xem các "ảnh" của ví dụ 2
shown = set()
for isl in find_islands(examples[1][0]):
    r0 = min(r for r, c in isl); r1 = max(r for r, c in isl)
    c0 = min(c for r, c in isl); c1 = max(c for r, c in isl)
    canvas = [["."] * (c1 - c0 + 1) for _ in range(r1 - r0 + 1)]
    for r, c in isl:
        canvas[r - r0][c - c0] = "#"
    pic = tuple("".join(row) for row in canvas)
    if pic not in shown:
        shown.add(pic)
        print("\n".join(pic)); print()

-> 1 | kỳ vọng 1
-> 3 | kỳ vọng 3
##
#.

##

.#
##



**Code nộp LeetCode (Cách 4):**

```python
class Solution:
    def numDistinctIslands(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        pictures = set()
        for si in range(m):
            for sj in range(n):
                if grid[si][sj] != 1:
                    continue
                grid[si][sj] = 0
                stack, cells = [(si, sj)], []
                while stack:
                    i, j = stack.pop()
                    cells.append((i, j))
                    for dx, dy in ((-1, 0), (0, 1), (1, 0), (0, -1)):
                        x, y = i + dx, j + dy
                        if 0 <= x < m and 0 <= y < n and grid[x][y] == 1:
                            grid[x][y] = 0
                            stack.append((x, y))
                r0 = min(r for r, c in cells); r1 = max(r for r, c in cells)
                c0 = min(c for r, c in cells); c1 = max(c for r, c in cells)
                canvas = [["."] * (c1 - c0 + 1) for _ in range(r1 - r0 + 1)]
                for r, c in cells:
                    canvas[r - r0][c - c0] = "#"
                pictures.add(tuple("".join(row) for row in canvas))
        return len(pictures)
```

**Độ phức tạp:** mỗi đảo tạo ảnh kích thước `bounding box` có thể lớn hơn số ô của đảo (đảo dạng đường chéo rộng `k x k` chỉ có `~2k` ô nhưng ảnh `k^2`). Tệ nhất Time/Space `O(m * n)` cho mỗi đảo, nhưng tổng ảnh thường nhỏ; với `50 x 50` vẫn nhẹ.

### 7.4. Cách 5: Union-Find rồi chuẩn hóa từng nhóm

**Ý tưởng:** thay DFS/BFS bằng **Union-Find** để gom các ô đất thành nhóm (đảo): với mỗi ô đất, gộp với ô đất **bên phải** và **bên dưới** (đủ để phủ mọi cặp kề, vì cặp trái/trên được tính từ phía ô kia). Sau đó gom các ô theo **gốc** của nhóm, rồi chuẩn hóa từng nhóm như Cách 2.

**Giải thích từng đoạn:**

1. Đánh chỉ số ô `(i, j)` thành `i * n + j` để dùng mảng `parent`.
2. Với mỗi ô đất: nếu ô bên phải là đất thì `union`; nếu ô bên dưới là đất thì `union`.
3. `groups[find(id)].append((i, j))`; mỗi nhóm là một đảo.
4. Chuẩn hóa mỗi nhóm bằng cách trừ ô nhỏ nhất theo `(hàng, cột)`, tạo `tuple` đã sắp xếp.

In [15]:
def union_find_shapes(grid):
    m, n = len(grid), len(grid[0])
    parent = list(range(m * n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    for i in range(m):
        for j in range(n):
            if grid[i][j]:
                if j + 1 < n and grid[i][j + 1]:
                    union(i * n + j, i * n + j + 1)
                if i + 1 < m and grid[i + 1][j]:
                    union(i * n + j, (i + 1) * n + j)

    groups = {}
    for i in range(m):
        for j in range(n):
            if grid[i][j]:
                groups.setdefault(find(i * n + j), []).append((i, j))

    shapes = set()
    for cells in groups.values():
        cells.sort()
        r0 = min(r for r, c in cells)
        c0 = min(c for r, c in cells)
        shapes.add(tuple((r - r0, c - c0) for r, c in cells))
    return len(shapes)

for grid, expected in examples:
    got = union_find_shapes(grid)
    print("->", got, "| kỳ vọng", expected)
    assert got == expected

-> 1 | kỳ vọng 1
-> 3 | kỳ vọng 3


**Code nộp LeetCode (Cách 5):**

```python
class Solution:
    def numDistinctIslands(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        parent = list(range(m * n))

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        def union(a, b):
            ra, rb = find(a), find(b)
            if ra != rb:
                parent[rb] = ra

        for i in range(m):
            for j in range(n):
                if grid[i][j]:
                    if j + 1 < n and grid[i][j + 1]:
                        union(i * n + j, i * n + j + 1)
                    if i + 1 < m and grid[i + 1][j]:
                        union(i * n + j, (i + 1) * n + j)

        groups = {}
        for i in range(m):
            for j in range(n):
                if grid[i][j]:
                    groups.setdefault(find(i * n + j), []).append((i, j))

        shapes = set()
        for cells in groups.values():
            cells.sort()
            r0 = min(r for r, c in cells)
            c0 = min(c for r, c in cells)
            shapes.add(tuple((r - r0, c - c0) for r, c in cells))
        return len(shapes)
```

**Độ phức tạp:** Union-Find với nén đường đi gần như `O(1)` mỗi thao tác (hằng số Ackermann ngược); Time `O(m * n * α)`, thực tế coi `O(m * n)` (cộng sắp xếp mỗi nhóm `O(S log S)` tệ nhất). Space `O(m * n)`. Ưu điểm: không đệ quy nên không lo sâu; không sửa lưới gốc.

## 8. TRACE từng bước

### 8.1. Trace DFS chuỗi đường đi cho đảo "L" của ví dụ 2

Đảo A: `(0,0), (0,1), (1,0)`. Mã hướng: `1` trên, `2` phải, `3` dưới, `4` trái; ô bắt đầu mã `0`. Thứ tự thử: trên, phải, dưới, trái.

| Bước | Hành động | `path` |
|---|---|---|
| 1 | vào `(0,0)` với `k = 0`: thêm `0` | `0` |
| 2 | thử trên: ra ngoài. Thử phải `(0,1)`: là đất → vào với `k = 2`: thêm `2` | `02` |
| 3 | tại `(0,1)`: trên, phải, dưới, trái đều không phải đất mới (hoặc đã thăm) → xong; thêm `-2` | `02-2` |
| 4 | về `(0,0)`: thử dưới `(1,0)`: đất → vào với `k = 3`: thêm `3` | `02-23` |
| 5 | tại `(1,0)`: không có ô mới; thêm `-3` | `02-23-3` |
| 6 | về `(0,0)`: thử trái: ra ngoài. Xong; thêm `-0` (= `0`) | `02-23-30` |

Dấu vân tay của A: `"02-23-30"`.

In [16]:
def trace_path(grid, si, sj):
    m, n = len(grid), len(grid[0])
    g = copy.deepcopy(grid)
    path = []
    def dfs(i, j, k):
        indent = "  " * len(stack)
        g[i][j] = 0
        path.append(str(k))
        print(f"{indent}vào ({i},{j}) mã {k:2d} -> path = {''.join(path)}")
        stack.append((i, j))
        for h in range(1, 5):
            x, y = i + dirs[h - 1], j + dirs[h]
            if 0 <= x < m and 0 <= y < n and g[x][y]:
                print(f"{indent}  hướng {names[h]} -> ({x},{y}) là đất mới, đi vào")
                dfs(x, y, h)
            else:
                print(f"{indent}  hướng {names[h]} -> ({x},{y}) không dùng được")
        stack.pop()
        path.append(str(-k))
        print(f"{indent}rời ({i},{j}) quay lui {-k:2d} -> path = {''.join(path)}")
    stack = []
    dfs(si, sj, 0)
    print("dấu vân tay:", "".join(path))

trace_path(examples[1][0], 0, 0)

vào (0,0) mã  0 -> path = 0
  hướng trên -> (-1,0) không dùng được
  hướng phải -> (0,1) là đất mới, đi vào
  vào (0,1) mã  2 -> path = 02
    hướng trên -> (-1,1) không dùng được
    hướng phải -> (0,2) không dùng được
    hướng dưới -> (1,1) không dùng được
    hướng trái -> (0,0) không dùng được
  rời (0,1) quay lui -2 -> path = 02-2
  hướng dưới -> (1,0) là đất mới, đi vào
  vào (1,0) mã  3 -> path = 02-23
    hướng trên -> (0,0) không dùng được
    hướng phải -> (1,1) không dùng được
    hướng dưới -> (2,0) không dùng được
    hướng trái -> (1,-1) không dùng được
  rời (1,0) quay lui -3 -> path = 02-23-3
  hướng trái -> (0,-1) không dùng được
rời (0,0) quay lui  0 -> path = 02-23-30
dấu vân tay: 02-23-30


Hãy xem dấu vân tay của các đảo còn lại và so sánh: hai đoạn ngang B và D phải cho **cùng** một chuỗi.

In [17]:
def all_signatures(grid):
    m, n = len(grid), len(grid[0])
    g = copy.deepcopy(grid)
    out, path = [], []
    def dfs(i, j, k):
        g[i][j] = 0
        path.append(str(k))
        for h in range(1, 5):
            x, y = i + dirs[h - 1], j + dirs[h]
            if 0 <= x < m and 0 <= y < n and g[x][y]:
                dfs(x, y, h)
        path.append(str(-k))
    for i in range(m):
        for j in range(n):
            if g[i][j]:
                start = (i, j)
                dfs(i, j, 0)
                out.append((start, "".join(path)))
                path.clear()
    return out

sigs = all_signatures(examples[1][0])
for start, sig in sigs:
    print("đảo bắt đầu tại", start, "->", sig)
print("số chuỗi khác nhau:", len({s for _, s in sigs}))

đảo bắt đầu tại (0, 0) -> 02-23-30
đảo bắt đầu tại (0, 3) -> 02-20
đảo bắt đầu tại (2, 4) -> 034-4-30
đảo bắt đầu tại (3, 0) -> 02-20
số chuỗi khác nhau: 3


### 8.2. Trace Cách 2 (tọa độ tương đối) cho ví dụ 2

| Đảo | Các ô (hàng, cột) | min hàng, min cột | Sau chuẩn hóa |
|---|---|---|---|
| A | `(0,0) (0,1) (1,0)` | 0, 0 | `{(0,0),(0,1),(1,0)}` |
| B | `(0,3) (0,4)` | 0, 3 | `{(0,0),(0,1)}` |
| C | `(2,4) (3,3) (3,4)` | 2, 3 | `{(0,1),(1,0),(1,1)}` |
| D | `(3,0) (3,1)` | 3, 0 | `{(0,0),(0,1)}` |

Số `frozenset` khác nhau: 3.

In [18]:
def trace_normalize(grid):
    shapes = {}
    for idx, isl in enumerate(find_islands(grid)):
        r0 = min(r for r, c in isl)
        c0 = min(c for r, c in isl)
        shape = frozenset((r - r0, c - c0) for r, c in isl)
        new = shape not in shapes
        shapes.setdefault(shape, idx)
        print(f"đảo {idx}: ô {isl} | trừ ({r0},{c0}) -> {sorted(shape)} | {'dạng MỚI' if new else 'trùng dạng đảo ' + str(shapes[shape])}")
    print("số dạng khác nhau:", len(shapes))

trace_normalize(examples[1][0])

đảo 0: ô [(0, 0), (0, 1), (1, 0)] | trừ (0,0) -> [(0, 0), (0, 1), (1, 0)] | dạng MỚI
đảo 1: ô [(0, 3), (0, 4)] | trừ (0,3) -> [(0, 0), (0, 1)] | dạng MỚI
đảo 2: ô [(2, 4), (3, 3), (3, 4)] | trừ (2,3) -> [(0, 1), (1, 0), (1, 1)] | dạng MỚI
đảo 3: ô [(3, 0), (3, 1)] | trừ (3,0) -> [(0, 0), (0, 1)] | trùng dạng đảo 1
số dạng khác nhau: 3


## 9. Code nộp LeetCode

Ta nộp **Cách 3** (DFS + chuỗi đường đi có quay lui) — gọn, đúng kiểu "dấu vân tay hình dạng".

In [19]:
class Solution:
    def numDistinctIslands(self, grid: List[List[int]]) -> int:
        def dfs(i: int, j: int, k: int):
            grid[i][j] = 0                      # đánh dấu đã thăm
            path.append(str(k))                 # ghi hướng đi vào ô này
            dirs = (-1, 0, 1, 0, -1)
            for h in range(1, 5):
                x, y = i + dirs[h - 1], j + dirs[h]
                if 0 <= x < m and 0 <= y < n and grid[x][y]:
                    dfs(x, y, h)
            path.append(str(-k))                # ghi quay lui khi rời ô này

        paths = set()
        path = []
        m, n = len(grid), len(grid[0])
        for i, row in enumerate(grid):
            for j, x in enumerate(row):
                if x:
                    dfs(i, j, 0)
                    paths.add("".join(path))
                    path.clear()
        return len(paths)


sol = Solution()
for grid, expected in examples:
    got = sol.numDistinctIslands(copy.deepcopy(grid))
    print("->", got, "| kỳ vọng", expected)
    assert got == expected

-> 1 | kỳ vọng 1
-> 3 | kỳ vọng 3


## 10. Vì sao đúng

**Hai đảo giống nhau ⇒ cùng chuỗi.** Nếu đảo `B` là tịnh tiến của đảo `A`, ô bắt đầu của DFS (ô đầu tiên gặp khi quét theo hàng rồi cột) của `B` ứng với ô bắt đầu của `A` (phép tịnh tiến bảo toàn thứ tự hàng-cột). Thứ tự thử hướng cố định, và mọi kiểm tra "ô kề có là đất chưa thăm" cho cùng kết quả ở hai đảo (cấu trúc giống hệt, đảo là thành phần liên thông nên các ô ngoài đảo không ảnh hưởng: ô kề ngoài đảo luôn là nước hoặc đã thăm). Vậy DFS thực hiện cùng chuỗi hành động, cùng chuỗi ghi.

**Chuỗi giống nhau ⇒ hai đảo giống nhau.** Chuỗi gồm các token "vào theo hướng `h`" và "ra" tạo thành một **cây DFS có thứ tự** (như dấu ngoặc đóng mở). Từ chuỗi ta dựng lại được duy nhất cây DFS: bắt đầu tại gốc, đọc `h` thì đi một bước theo hướng `h` để tới ô con, đọc `-h` thì quay về ô cha. Cây DFS phủ toàn bộ ô của đảo, nên từ chuỗi ta phục hồi **toàn bộ tọa độ tương đối** của đảo so với ô bắt đầu. Hai chuỗi bằng nhau ⇒ hai đảo có cùng tập tọa độ tương đối ⇒ giống nhau qua tịnh tiến.

Nếu **bỏ ký hiệu quay lui** thì từ chuỗi không biết mỗi bước đi bắt đầu từ ô nào (ô cha ở đâu), nên phục hồi không còn duy nhất (ví dụ ở mục 6).

**Đánh dấu đã thăm** (gán `0`) bảo đảm mỗi ô vào DFS một lần, nên mỗi đảo chỉ được ghi một lần và các đảo khác nhau không lẫn.

Kiểm chứng bằng code: (a) khớp oracle trên rất nhiều lưới nhỏ ngẫu nhiên; (b) từ chuỗi đường đi **dựng lại** tập ô tương đối của đảo rồi so với chuẩn hóa tọa độ.

In [20]:
def decode_path(sig):
    """Từ chuỗi đường đi (có quay lui) dựng lại tập ô tương đối của đảo, xuất phát tại (0, 0)."""
    tokens, i = [], 0
    while i < len(sig):                    # tách token: số có thể có dấu '-'
        if sig[i] == "-":
            tokens.append(int(sig[i:i + 2])); i += 2
        else:
            tokens.append(int(sig[i])); i += 1
    pos, stack, cells = (0, 0), [], set()
    for t in tokens:
        if t >= 0:                         # vào ô theo hướng t (t = 0: ô bắt đầu)
            if t:
                pos = (pos[0] + dirs[t - 1], pos[1] + dirs[t])
            stack.append(t)
            cells.add(pos)
        else:                              # quay lui
            back = stack.pop()
            if back:
                pos = (pos[0] - dirs[back - 1], pos[1] - dirs[back])
    return frozenset(cells)

rng = random.Random(694)
checked = 0
for _ in range(400):
    m, n = rng.randint(1, 7), rng.randint(1, 7)
    grid = [[1 if rng.random() < 0.5 else 0 for _ in range(n)] for _ in range(m)]
    for sig_start, sig in all_signatures(grid):
        pass
    islands = find_islands(grid)
    sigs = all_signatures(grid)
    assert len(islands) == len(sigs)
    # đảo thứ t trong find_islands và all_signatures cùng thứ tự quét hàng-cột
    for isl, (start, sig) in zip(islands, sigs):
        assert isl[0] == start
        shape = decode_path(sig)
        r0 = min(r for r, c in isl); c0 = min(c for r, c in isl)
        want = frozenset((r - isl[0][0], c - isl[0][1]) for r, c in isl)   # tương đối so với ô bắt đầu
        assert shape == want, (isl, sig)
        checked += 1
print("giải mã chuỗi đường đi của", checked, "đảo: khôi phục đúng tọa độ tương đối")

giải mã chuỗi đường đi của

 1053 đảo: khôi phục đúng tọa độ tương đối


## 11. Time/Space Complexity

Gọi `m, n` là kích thước lưới, `S <= m * n` là số ô đất, `I` là số đảo.

| Cách | Time | Space |
|---|---|---|
| 1. So sánh từng cặp | `O(m*n + I * S)` | `O(m*n)` |
| 2. BFS + tọa độ tương đối + `set` | `O(m*n)` | `O(m*n)` |
| 3. DFS chuỗi đường đi có quay lui | `O(m*n)` | `O(m*n)` (gồm ngăn xếp đệ quy) |
| 4. Khung bao thành ảnh | `O(m*n)` + kích thước ảnh | `O(m*n)` đến `O(I * m * n)` tệ nhất |
| 5. Union-Find + chuẩn hóa | `O(m*n*α + S log S)` | `O(m*n)` |

Mọi ô được thăm **một lần** (cờ đã thăm hoặc gán `0`). Chuỗi/`frozenset` của mỗi đảo có kích thước `O(kích thước đảo)`, tổng `O(S)`. Với `50 x 50 = 2500` ô, các cách đều rất nhanh. Lưu ý đệ quy Python: đảo dài `2500` ô có thể sâu `2500` tầng, vượt giới hạn mặc định `1000`, nên ta tăng `sys.setrecursionlimit` (LeetCode thường đã đặt giới hạn đủ cao cho bài này).

## 12. Bộ test

Nhóm test: ví dụ đề; edge case (lưới `1 x 1`, toàn nước, toàn đất, bàn cờ, đảo chéo nhau không nối, xoay/lật là khác); phản ví dụ cho các cách làm sai; đối chiếu **5 cách + lớp nộp** với **oracle độc lập** (thử mọi phép tịnh tiến giữa từng cặp đảo) trên lưới ngẫu nhiên có seed; test lớn `50 x 50`.

Vì các hàm sửa lưới tại chỗ, ta luôn truyền **bản sao** (`copy.deepcopy`) cho mỗi lần gọi.

In [21]:
def oracle(grid):
    """Oracle: gom đảo bằng ngăn xếp; hai đảo 'giống nhau' nếu SOME phép tịnh tiến đưa tập ô này lên tập ô kia
    (thử mọi vector dịch đưa một ô bất kỳ của A lên ô đầu tiên của B). Không dùng mốc cố định hay chuỗi."""
    m, n = len(grid), len(grid[0])
    seen = set()
    islands = []
    for si in range(m):
        for sj in range(n):
            if grid[si][sj] == 1 and (si, sj) not in seen:
                comp, stack = set(), [(si, sj)]
                seen.add((si, sj))
                while stack:
                    i, j = stack.pop()
                    comp.add((i, j))
                    for di, dj in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                        x, y = i + di, j + dj
                        if 0 <= x < m and 0 <= y < n and grid[x][y] == 1 and (x, y) not in seen:
                            seen.add((x, y))
                            stack.append((x, y))
                islands.append(comp)

    def translate_equal(a, b):
        if len(a) != len(b):
            return False
        tb = next(iter(b))
        for ta in a:                                   # thử đưa từng ô của a lên ô tb của b
            dx, dy = tb[0] - ta[0], tb[1] - ta[1]
            if {(x + dx, y + dy) for x, y in a} == b:
                return True
        return False

    classes = []
    for isl in islands:
        if not any(translate_equal(isl, c) for c in classes):
            classes.append(isl)
    return len(classes)

methods = {
    "brute_force": brute_force,
    "normalized_set": normalized_set,
    "dfs_path": dfs_path,
    "bounding_box_picture": bounding_box_picture,
    "union_find_shapes": union_find_shapes,
    "Solution": Solution().numDistinctIslands,
}

def run_all(grid, expected=None):
    results = {}
    for name, fn in methods.items():
        results[name] = fn(copy.deepcopy(grid))
    if expected is not None:
        assert all(v == expected for v in results.values()), (grid, results, expected)
    return results

# 12.1 ví dụ của đề
for grid, expected in examples:
    run_all(grid, expected)
    assert oracle(copy.deepcopy(grid)) == expected
print("2 ví dụ: cả", len(methods), "cách và oracle đúng")

2 ví dụ: cả

 6 cách và oracle đúng


In [22]:
# 12.2 edge case
edge = [
    ([[0]], 0),                                   # một ô nước
    ([[1]], 1),                                   # một ô đất
    ([[0, 0, 0], [0, 0, 0]], 0),                  # toàn nước
    ([[1, 1, 1], [1, 1, 1]], 1),                  # toàn đất: một đảo
    ([[1, 0], [0, 1]], 1),                        # hai ô chéo nhau: hai đảo 1 ô, cùng hình -> 1 dạng
    ([[1, 0, 1, 0, 1]], 1),                       # ba đảo một ô giống nhau
    ([[1, 1, 0, 1, 0], [0, 0, 0, 1, 0]], 2),      # đoạn ngang 2 ô và đoạn dọc 2 ô: KHÁC (không xoay)
    ([[1, 1, 0, 0, 1, 1],
      [1, 0, 0, 0, 0, 1]], 2),                    # L và hình gương của L: KHÁC (không lật)
    ([[1, 1, 0, 1, 0],
      [0, 1, 0, 1, 1]], 2),                       # 'S'-ghép? hai tetromino khác nhau
    ([[1, 0, 1], [1, 0, 1], [0, 0, 0], [1, 1, 0]], 2),   # hai đoạn dọc 2 ô giống nhau + đoạn ngang
]
for grid, expected in edge:
    got = run_all(grid, expected)
    assert oracle(copy.deepcopy(grid)) == expected, (grid, oracle(copy.deepcopy(grid)))

# bàn cờ: mọi ô đất cô lập -> 1 dạng; 50x50 toàn đất -> 1 dạng
checker = [[(i + j) % 2 for j in range(50)] for i in range(50)]
run_all(checker, 1)
run_all([[1] * 50 for _ in range(50)], 1)
run_all([[0] * 50 for _ in range(50)], 0)
print("edge case (kể cả bàn cờ và 50x50 toàn đất/toàn nước): OK")

edge case (kể cả bàn cờ và 50x50 toàn đất/toàn nước): OK

In [23]:
# 12.3 phản ví dụ: các cách làm SAI phổ biến
def wrong_no_backtrack(grid):                    # chỉ ghi hướng đi vào, không ghi quay lui
    g = copy.deepcopy(grid)
    m, n = len(g), len(g[0])
    shapes, path = set(), []
    def dfs(i, j, k):
        g[i][j] = 0
        path.append(str(k))
        for h in range(1, 5):
            x, y = i + dirs[h - 1], j + dirs[h]
            if 0 <= x < m and 0 <= y < n and g[x][y]:
                dfs(x, y, h)
    for i in range(m):
        for j in range(n):
            if g[i][j]:
                dfs(i, j, 0)
                shapes.add("".join(path))
                path.clear()
    return len(shapes)

def wrong_absolute(grid):                        # không chuẩn hóa: dùng tọa độ tuyệt đối
    return len({frozenset(isl) for isl in find_islands(grid)})

def wrong_rotation(grid):                        # coi xoay/lật là giống (đó là bài 711, không phải 694)
    shapes = set()
    for isl in find_islands(grid):
        variants = []
        pts = isl
        for _ in range(4):
            pts = [(c, -r) for r, c in pts]            # xoay 90 độ
            for flip in (1, -1):
                q = [(r, c * flip) for r, c in pts]
                r0 = min(r for r, c in q); c0 = min(c for r, c in q)
                variants.append(tuple(sorted((r - r0, c - c0) for r, c in q)))
        shapes.add(min(variants))
    return len(shapes)

def wrong_diag(grid):                            # coi 8 hướng (cả chéo) là liên thông
    g = copy.deepcopy(grid)
    m, n = len(g), len(g[0])
    count = 0
    def dfs(i, j):
        g[i][j] = 0
        for di in (-1, 0, 1):
            for dj in (-1, 0, 1):
                x, y = i + di, j + dj
                if 0 <= x < m and 0 <= y < n and g[x][y]:
                    dfs(x, y)
    for i in range(m):
        for j in range(n):
            if g[i][j]:
                dfs(i, j)
                count += 1
    return count

# hai hình P và Q của mục 6 (cùng chuỗi nếu không quay lui)
g_pq = [[1, 1, 0, 1, 1],
        [1, 0, 0, 0, 1]]
print("P và Q khác hình:        đúng =", oracle(copy.deepcopy(g_pq)), "| không quay lui =", wrong_no_backtrack(g_pq))
g_abs = examples[0][0]
print("ví dụ 1 (2 hình vuông):  đúng =", oracle(copy.deepcopy(g_abs)), "| tọa độ tuyệt đối =", wrong_absolute(g_abs))
g_rot = [[1, 1, 0, 1, 0], [0, 0, 0, 1, 0]]
print("đoạn ngang và đoạn dọc:  đúng =", oracle(copy.deepcopy(g_rot)), "| coi xoay là giống =", wrong_rotation(g_rot))
g_dg = [[1, 0], [0, 1]]
print("hai ô chéo nhau:         đúng số dạng =", oracle(copy.deepcopy(g_dg)), "(2 đảo) | coi chéo là nối: số đảo =", wrong_diag(g_dg))
assert wrong_no_backtrack(g_pq) != oracle(copy.deepcopy(g_pq))
assert wrong_absolute(g_abs) != oracle(copy.deepcopy(g_abs))
assert wrong_rotation(g_rot) != oracle(copy.deepcopy(g_rot))
print("các cách sai bị phát hiện")

P và Q khác hình:        đúng =

 2 | không quay lui = 1
ví dụ 1 (2 hình vuông):  đúng = 1 | tọa độ tuyệt đối = 2
đoạn ngang và đoạn dọc:  đúng = 2 | coi xoay là giống = 1
hai ô chéo nhau:         đúng số dạng = 1 (2 đảo) | coi chéo là nối: số đảo = 1
các cách sai bị phát hiện


In [24]:
# 12.4 đối chiếu mọi cách với oracle trên lưới ngẫu nhiên nhỏ (seed cố định)
rng = random.Random(20240694)
cases = 0
for _ in range(1500):
    m, n = rng.randint(1, 8), rng.randint(1, 8)
    p = rng.choice([0.25, 0.4, 0.5, 0.65])
    grid = [[1 if rng.random() < p else 0 for _ in range(n)] for _ in range(m)]
    expected = oracle(copy.deepcopy(grid))
    run_all(grid, expected)
    cases += 1
print("đối chiếu", cases, "lưới ngẫu nhiên với", len(methods), "cách: OK")

# lưới sinh theo 'hình mẫu lặp' để chắc chắn có nhiều đảo giống nhau
shapes = [[(0, 0), (0, 1), (1, 0)], [(0, 0), (1, 0), (2, 0)], [(0, 0), (0, 1), (1, 1)], [(0, 0)], [(0, 0), (0, 1), (0, 2), (1, 1)]]
cnt = 0
for _ in range(300):
    m, n = 14, 14
    grid = [[0] * n for _ in range(m)]
    placed = 0
    for _ in range(12):
        sh = rng.choice(shapes)
        r, c = rng.randint(0, m - 4), rng.randint(0, n - 4)
        cells = [(r + a, c + b) for a, b in sh]
        # chỉ đặt khi không chạm (kể cả kề cạnh) đảo khác
        ok = all(grid[x + dx][y + dy] == 0
                 for x, y in cells for dx, dy in ((0, 0), (1, 0), (-1, 0), (0, 1), (0, -1))
                 if 0 <= x + dx < m and 0 <= y + dy < n)
        if ok:
            for x, y in cells:
                grid[x][y] = 1
            placed += 1
    expected = oracle(copy.deepcopy(grid))
    run_all(grid, expected)
    cnt += 1
print("đối chiếu", cnt, "lưới ghép hình mẫu lặp (14x14): OK")

đối chiếu 1500 lưới ngẫu nhiên với 6 cách: OK


đối chiếu 300 lưới ghép hình mẫu lặp (14x14): OK


In [25]:
# 12.5 test lớn đúng ràng buộc: 50 x 50 (oracle chậm nên đối chiếu các cách với nhau và với Cách 1)
rng = random.Random(5050)
fast = ["normalized_set", "dfs_path", "bounding_box_picture", "union_find_shapes", "Solution", "brute_force"]
for p in (0.1, 0.3, 0.5, 0.6, 0.9):
    grid = [[1 if rng.random() < p else 0 for _ in range(50)] for _ in range(50)]
    t0 = time.perf_counter()
    results = {name: methods[name](copy.deepcopy(grid)) for name in fast}
    dt = time.perf_counter() - t0
    assert len(set(results.values())) == 1, results
    print(f"mật độ đất {p:.1f}: {results['Solution']:3d} dạng đảo khác nhau  ({dt * 1000:.0f} ms cho 6 cách)")

# đảo rắn dài nhất: DFS sâu nhất (đường ngoằn ngoèo phủ gần hết lưới)
snake = [[0] * 49 for _ in range(49)]
for i in range(0, 49, 2):
    for j in range(49):
        snake[i][j] = 1
    if i + 1 < 49:
        snake[i + 1][48 if (i // 2) % 2 == 0 else 0] = 1
res = run_all(snake, 1)
print("đảo rắn 49x49 (DFS sâu ~1200 tầng): ", res["Solution"], "dạng; mọi cách khớp")

mật độ đất 0.1:  10 dạng đảo khác nhau  (26 ms cho 6 cách)
mật độ đất 0.3:  58 dạng đảo khác nhau  (26 ms cho 6 cách)
mật độ đất 0.5:  75 dạng đảo khác nhau  (19 ms cho 6 cách)
mật độ đất 0.6:  28 dạng đảo khác nhau  (20 ms cho 6 cách)
mật độ đất 0.9:   1 dạng đảo khác nhau  (36 ms cho 6 cách)
đảo rắn 49x49 (DFS sâu ~1200 tầng):  1 dạng; mọi cách khớp


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ ghi hướng đi, quên ký hiệu quay lui

Hai hình **khác nhau** có thể cho **cùng chuỗi** nếu không ghi quay lui (xem mục 6).

In [26]:
print("hai hình P, Q khác nhau, oracle =", oracle(copy.deepcopy(g_pq)), "dạng")
print("SAI  (không quay lui):", wrong_no_backtrack(g_pq))
print("ĐÚNG                 :", Solution().numDistinctIslands(copy.deepcopy(g_pq)))

hai hình P, Q khác nhau, oracle = 2 dạng
SAI  (không quay lui): 1
ĐÚNG                 : 2


**Vì sao sai:** chuỗi "0 phải xuống" không cho biết ô "xuống" kề với ô nào. **Sửa:** thêm `path.append(str(-k))` sau khi thử xong 4 hướng của ô.

### Lỗi 2: Dùng tọa độ tuyệt đối

Nếu bỏ `frozenset(isl)` thẳng vào `set` mà không trừ mốc, hai đảo giống hệt nhưng ở chỗ khác nhau bị tính là khác nhau.

In [27]:
print("ví dụ 1: SAI  :", wrong_absolute(examples[0][0]), "(đúng 1)")
print("ví dụ 1: ĐÚNG :", Solution().numDistinctIslands(copy.deepcopy(examples[0][0])))

ví dụ 1: SAI  : 2 (đúng 1)
ví dụ 1: ĐÚNG : 1


**Vì sao sai:** hai hình vuông `2x2` có tọa độ tuyệt đối khác nhau. **Sửa:** trừ đi hàng nhỏ nhất và cột nhỏ nhất (hoặc dùng chuỗi đường đi bắt đầu từ ô đầu).

### Lỗi 3: Đưa `list` vào `set`

`set` yêu cầu phần tử hashable; `list` thì không.

In [28]:
shapes = set()
cells = [(0, 0), (0, 1)]
try:
    shapes.add(cells)                      # SAI
except TypeError as e:
    print("SAI  : TypeError:", e)
shapes.add(tuple(cells))                   # ĐÚNG: tuple
shapes.add(frozenset(cells))               # hoặc frozenset
print("ĐÚNG:", len(shapes), "phần tử")

SAI  : TypeError: unhashable type: 'list'
ĐÚNG: 2 phần tử


**Vì sao sai:** `list` có thể thay đổi nên không có mã băm. **Sửa:** `tuple(...)`, `frozenset(...)` hoặc chuỗi `"".join(...)`.

### Lỗi 4: Coi xoay hoặc lật là giống nhau

Đề chỉ cho phép **tịnh tiến**. Đoạn ngang 2 ô và đoạn dọc 2 ô là **hai dạng khác nhau** (đó là cách bài LeetCode 711 phức tạp hơn khi cho xoay/lật).

In [29]:
g = [[1, 1, 0, 1, 0], [0, 0, 0, 1, 0]]       # đoạn ngang + đoạn dọc
print("SAI  (coi xoay là giống):", wrong_rotation(g))
print("ĐÚNG                    :", Solution().numDistinctIslands(copy.deepcopy(g)))

SAI  (coi xoay là giống): 1
ĐÚNG                    : 2


**Vì sao sai:** nếu chuẩn hóa bằng cách chọn dạng nhỏ nhất trong các phép xoay/lật thì hai đảo xoay của nhau bị gộp làm một. **Sửa:** chỉ chuẩn hóa bằng tịnh tiến.

### Lỗi 5: Dùng lưới đã bị sửa cho lần chạy tiếp theo

DFS gán `0` để đánh dấu, nên sau khi chạy, lưới gốc bị phá. Nếu bạn chạy hai cách liên tiếp trên cùng một lưới, cách thứ hai thấy lưới toàn nước.

In [30]:
g = copy.deepcopy(examples[1][0])
first = Solution().numDistinctIslands(g)       # sửa g tại chỗ
second = Solution().numDistinctIslands(g)      # g giờ toàn 0
print("lần 1:", first, "| lần 2 trên cùng lưới:", second, "(sai vì lưới đã bị xóa)")
assert (first, second) == (3, 0)

lần 1: 3 | lần 2 trên cùng lưới: 0 (sai vì lưới đã bị xóa)


**Vì sao sai:** thao tác "đánh dấu đã thăm" thay đổi dữ liệu đầu vào. **Sửa:** khi test hãy `copy.deepcopy`; trong lời giải có thể dùng mảng `seen` riêng nếu cần giữ nguyên lưới (như `find_islands`).

## 14. Mẹo & nhận dạng pattern

**Dấu hiệu nhận ra:**

- Bài lưới đếm "thành phần liên thông" (đảo): **DFS/BFS/Union-Find**.
- Thêm yêu cầu "**hình dạng giống nhau**" (đếm khác nhau): cần **dấu vân tay (canonical form)** bất biến theo phép biến đổi cho phép.
- "Tịnh tiến": chuẩn hóa bằng **trừ mốc** hoặc **chuỗi đường đi tương đối**.

**Mẹo:**

1. Chọn mốc **không phụ thuộc vị trí**: ô đầu theo thứ tự quét, hoặc `(min hàng, min cột)`.
2. Với chuỗi đường đi: luôn ghi cả **vào** và **ra** (quay lui), và **cố định thứ tự thử hướng**.
3. Dấu vân tay phải **hashable**: chuỗi, tuple, frozenset.
4. Nếu cho phép xoay/lật (bài 711): tạo **8 biến thể**, chuẩn hóa từng cái, lấy **dạng nhỏ nhất**.
5. Đánh dấu đã thăm bằng cách sửa lưới là nhanh, nhưng **sửa dữ liệu vào**; nhớ sao chép khi test.
6. Cẩn thận độ sâu đệ quy: đảo dài `2500` ô; dùng BFS/stack nếu cần.

## 15. Biến thể

### 15.1. LeetCode 711 — Distinct Islands II (cho phép xoay và lật)

Mỗi đảo có tối đa **8 biến thể** (4 phép xoay, mỗi cái có thêm bản lật gương). Chuẩn hóa từng biến thể bằng tịnh tiến rồi lấy biến thể **nhỏ nhất** (theo thứ tự tuple) làm dấu vân tay chung.

In [31]:
def canonical_with_symmetry(cells):
    variants = []
    pts = list(cells)
    for _ in range(4):
        pts = [(c, -r) for r, c in pts]                  # xoay 90 độ
        for flip in (1, -1):
            q = [(r, c * flip) for r, c in pts]          # có/không lật gương
            r0 = min(r for r, c in q); c0 = min(c for r, c in q)
            variants.append(tuple(sorted((r - r0, c - c0) for r, c in q)))
    return min(variants)

def distinct_islands_ii(grid):
    return len({canonical_with_symmetry(isl) for isl in find_islands(grid)})

# L và gương của L; đoạn ngang và đoạn dọc
print(distinct_islands_ii([[1, 1, 0, 0, 1, 1], [1, 0, 0, 0, 0, 1]]))   # 1 (gương của nhau)
print(distinct_islands_ii([[1, 1, 0, 1, 0], [0, 0, 0, 1, 0]]))          # 1 (xoay của nhau)
print(distinct_islands_ii(examples[1][0]))                             # 2: A và C là xoay/lật của nhau nên gộp lại

1
1
2


Ở ví dụ 2, đảo A (hình "L" có 3 ô) và C (hình "┘" có 3 ô) là **xoay/lật của nhau** (cả hai là tromino chữ L), nên với bài 711 chỉ còn **2** dạng (chữ L, đoạn 2 ô). Kết quả in ra phụ thuộc đúng như vậy; nếu bạn thấy `2` thay vì `3` ở dòng cuối thì đó là hành vi đúng của bài 711.

### 15.2. Liệt kê các dạng thay vì chỉ đếm

Chỉ cần trả về tập dấu vân tay (hoặc một đảo đại diện cho mỗi dạng): thay `len(set)` bằng việc giữ `dict` từ dấu vân tay tới đảo đầu tiên gặp.

In [32]:
def one_representative_per_shape(grid):
    reps = {}
    for isl in find_islands(grid):
        r0 = min(r for r, c in isl); c0 = min(c for r, c in isl)
        key = frozenset((r - r0, c - c0) for r, c in isl)
        reps.setdefault(key, isl)
    return list(reps.values())

for isl in one_representative_per_shape(examples[1][0]):
    print(isl)

[(0, 0), (0, 1), (1, 0)]
[(0, 3), (0, 4)]
[(2, 4), (3, 3), (3, 4)]


### 15.3. Các bài cùng họ

- **LeetCode 200 — Number of Islands:** chỉ đếm đảo (không quan tâm hình dạng).
- **LeetCode 695 — Max Area of Island:** diện tích đảo lớn nhất.
- **LeetCode 711 — Number of Distinct Islands II:** cho phép xoay/lật (15.1).
- **LeetCode 463 — Island Perimeter:** chu vi đảo.
- **LeetCode 1905 — Count Sub Islands:** đảo của lưới 2 nằm gọn trong đảo của lưới 1.
- **LeetCode 1254 — Number of Closed Islands:** đảo không chạm mép.
- **LeetCode 305 — Number of Islands II:** thêm đất dần, dùng Union-Find (khai thác Cách 5).

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. So sánh từng cặp đảo | so khớp hiệu tọa độ với từng đại diện | `O(m*n + I*S)` | `O(m*n)` | baseline, hiểu đề |
| 2. BFS + tọa độ tương đối + `set` | trừ min hàng/cột, `frozenset` | `O(m*n)` | `O(m*n)` | rõ ràng, dễ mở rộng (xoay/lật) |
| 3. DFS chuỗi đường đi có quay lui | chuỗi mã hướng vào/ra | `O(m*n)` | `O(m*n)` | **cách nộp chuẩn**, ngắn gọn |
| 4. Khung bao thành ảnh | cắt hình chữ nhật nhỏ nhất, `#`/`.` | `O(m*n)` + ảnh | `O(m*n)` | trực quan, dễ in ra kiểm tra |
| 5. Union-Find + chuẩn hóa | gom nhóm bằng DSU rồi trừ mốc | `O(m*n*α + S log S)` | `O(m*n)` | không đệ quy, không sửa lưới; bài lưới thay đổi dần |

Kết luận: để thi dùng Cách 3 (hoặc Cách 2 nếu muốn dễ chứng minh); biết Cách 2 thì chuyển sang 711 rất tự nhiên.

## 17. Bài tập vận dụng

Làm từ dễ đến khó. Đáp án ở **mục 19**. Các cell TODO **chưa được gọi**; bạn tự điền rồi tự chạy.

### Bài 1 (đoán output)

Quy ước mã hướng: `1` trên, `2` phải, `3` dưới, `4` trái, bắt đầu `0`, có quay lui. Đảo gồm các ô `(0,0), (0,1), (0,2)` (đoạn ngang 3 ô). Chuỗi đường đi (nối không dấu cách) là gì?

### Bài 2 (tính tay)

Với lưới:

```
1 1 0 0 1
0 1 0 0 1
0 0 0 0 0
1 0 1 1 0
1 0 0 0 0
```

Liệt kê các đảo, chuẩn hóa từng đảo, và cho biết có bao nhiêu dạng khác nhau.

### Bài 3 (điền TODO)

Điền chỗ trống của Cách 2 (chuẩn hóa tọa độ):

In [33]:
def ex3_normalize(grid):
    shapes = set()
    for isl in find_islands(grid):                 # mỗi đảo: list ô (hàng, cột)
        r0 = ...                                   # TODO 1: hàng nhỏ nhất của đảo
        c0 = ...                                   # TODO 2: cột nhỏ nhất của đảo
        shapes.add(frozenset((r - ..., c - ...) for r, c in isl))   # TODO 3: trừ mốc
    return len(shapes)

# Khi đã điền xong, bỏ dấu # ở dòng sau để thử:
# assert ex3_normalize([[1, 1, 0, 1, 1], [1, 0, 0, 0, 0], [0, 0, 0, 0, 1], [1, 1, 0, 1, 1]]) == 3

### Bài 4 (tự viết lại)

Viết lại **Cách 3 (DFS chuỗi đường đi có quay lui)** từ đầu, không nhìn bài giảng. Nhớ: đánh dấu thăm, ghi mã hướng vào, thử 4 hướng, ghi quay lui `-k`, quét lưới và bỏ chuỗi vào `set`.

In [34]:
def ex4_dfs_path(grid):
    # TODO: tự viết DFS chuỗi đường đi có quay lui, trả về số dạng đảo khác nhau
    ...

# Khi đã viết xong, bỏ dấu # để thử:
# assert ex4_dfs_path([[1, 1, 0, 1, 1], [1, 0, 0, 0, 0], [0, 0, 0, 0, 1], [1, 1, 0, 1, 1]]) == 3

### Bài 5 (mini challenge, biến thể)

Viết `distinct_islands_ii(grid)` (LeetCode 711): hai đảo giống nhau nếu một đảo có thể **xoay (90 độ) và/hoặc lật gương** rồi **tịnh tiến** để trùng đảo kia. Gợi ý: với mỗi đảo tạo 8 biến thể, chuẩn hóa, lấy nhỏ nhất. Kiểm tra: xoay hoặc lật **cả lưới** không đổi đáp án.

In [35]:
def ex5_distinct_ii(grid):
    # TODO: đếm số đảo khác nhau, cho phép xoay và lật
    ...

# Khi đã viết xong, thử bằng test ở mục 19

## 18. Gợi ý 3 mức

**Bài 3**

- Mức 1 (nhẹ): để "tịnh tiến về gốc" ta trừ đi gì trên mỗi tọa độ?
- Mức 2 (vừa): trừ **hàng nhỏ nhất** khỏi hàng và **cột nhỏ nhất** khỏi cột; dùng `min` trên các hàng/cột của đảo.
- Mức 3 (gần đáp án): `r0 = min(r for r, c in isl)`; `c0 = min(c for r, c in isl)`; `(r - r0, c - c0)`.

**Bài 4**

- Mức 1: DFS nhận `(i, j, k)`; `k` là mã hướng vừa đi tới ô này (bắt đầu `0`).
- Mức 2: vào ô: gán `0`, `path.append(str(k))`; thử 4 hướng; rời ô: `path.append(str(-k))`.
- Mức 3: ngoài DFS, vòng quét `for i, for j: if grid[i][j]: dfs(i, j, 0); paths.add("".join(path)); path.clear()`.

**Bài 5**

- Mức 1: xoay 90 độ biến `(r, c)` thành `(c, -r)`; lật gương biến `(r, c)` thành `(r, -c)`.
- Mức 2: lặp 4 lần xoay, mỗi lần thử có/không lật: 8 biến thể; chuẩn hóa mỗi biến thể bằng tịnh tiến, sắp xếp thành `tuple`.
- Mức 3: dấu vân tay = `min` của 8 tuple đó; đếm số dấu vân tay khác nhau bằng `set`.

## 19. Đáp án tham khảo

**Bài 1:** bắt đầu `0`; sang phải vào `(0,1)` mã `2`; từ `(0,1)` thử trên (ra ngoài), phải vào `(0,2)` mã `2`; từ `(0,2)` không đi tiếp được, quay lui `-2`; về `(0,1)` thử dưới, trái (đã thăm), quay lui `-2`; về `(0,0)` quay lui `0`. Chuỗi: `0` `2` `2` `-2` `-2` `0` → **`022-2-20`**.

**Bài 2:** các đảo: `{(0,0),(0,1),(1,1)}` (hình "┐"), `{(0,4),(1,4)}` (đoạn dọc 2 ô), `{(3,0),(4,0)}` (đoạn dọc 2 ô), `{(3,2),(3,3)}` (đoạn ngang 2 ô). Chuẩn hóa: `{(0,0),(0,1),(1,1)}`, `{(0,0),(1,0)}`, `{(0,0),(1,0)}`, `{(0,0),(0,1)}`. Có **3** dạng khác nhau (hai đoạn dọc trùng nhau).

In [36]:
# Bài 1
tokens = ["0", "2", "2", "-2", "-2", "0"]
sig = "".join(tokens)
assert sig == "022-2-20"
_, real_sig = all_signatures([[1, 1, 1]])[0]
assert real_sig == sig
print("Bài 1:", sig)

# Bài 2
g2 = [[1, 1, 0, 0, 1],
      [0, 1, 0, 0, 1],
      [0, 0, 0, 0, 0],
      [1, 0, 1, 1, 0],
      [1, 0, 0, 0, 0]]
assert len(find_islands(g2)) == 4
assert Solution().numDistinctIslands(copy.deepcopy(g2)) == 3 == oracle(copy.deepcopy(g2))
print("Bài 2: 4 đảo, 3 dạng")

# Bài 3
def ex3_answer(grid):
    shapes = set()
    for isl in find_islands(grid):
        r0 = min(r for r, c in isl)
        c0 = min(c for r, c in isl)
        shapes.add(frozenset((r - r0, c - c0) for r, c in isl))
    return len(shapes)

# Bài 4
def ex4_answer(grid):
    g = copy.deepcopy(grid)
    m, n = len(g), len(g[0])
    paths, path = set(), []
    def dfs(i, j, k):
        g[i][j] = 0
        path.append(str(k))
        for h in range(1, 5):
            x, y = i + dirs[h - 1], j + dirs[h]
            if 0 <= x < m and 0 <= y < n and g[x][y]:
                dfs(x, y, h)
        path.append(str(-k))
    for i in range(m):
        for j in range(n):
            if g[i][j]:
                dfs(i, j, 0)
                paths.add("".join(path))
                path.clear()
    return len(paths)

# Bài 5
def ex5_answer(grid):
    def canon(cells):
        best = None
        pts = list(cells)
        for _ in range(4):
            pts = [(c, -r) for r, c in pts]
            for flip in (1, -1):
                q = [(r, c * flip) for r, c in pts]
                r0 = min(r for r, c in q); c0 = min(c for r, c in q)
                t = tuple(sorted((r - r0, c - c0) for r, c in q))
                if best is None or t < best:
                    best = t
        return best
    return len({canon(isl) for isl in find_islands(grid)})

def rotate_grid(g):                 # xoay cả lưới 90 độ
    return [list(row) for row in zip(*g[::-1])]
def mirror_grid(g):                 # lật gương cả lưới
    return [row[::-1] for row in g]

rng = random.Random(19)
for _ in range(400):
    m, n = rng.randint(1, 8), rng.randint(1, 8)
    grid = [[1 if rng.random() < 0.45 else 0 for _ in range(n)] for _ in range(m)]
    want = oracle(copy.deepcopy(grid))
    assert ex3_answer(grid) == ex4_answer(grid) == want
    ii = ex5_answer(grid)
    assert ii <= want                                    # cho phép xoay/lật chỉ có thể gộp thêm dạng
    assert ii == ex5_answer(rotate_grid(grid)) == ex5_answer(mirror_grid(grid))   # bất biến khi xoay/lật cả lưới
assert ex5_answer([[1, 1, 0, 1, 0], [0, 0, 0, 1, 0]]) == 1
assert ex5_answer(examples[1][0]) == 2
print("Bài 3, 4: khớp oracle; Bài 5: gộp được xoay/lật và bất biến khi xoay/lật cả lưới (400 lưới)")

Bài 1: 022-2-20
Bài 2: 4 đảo, 3 dạng


Bài 3, 4: khớp oracle; Bài 5: gộp được xoay/lật và bất biến khi xoay/lật cả lưới (400 lưới)


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "đếm hình dạng khác nhau của đảo" (DFS + chuỗi đường đi):**

```python
class Solution:
    def numDistinctIslands(self, grid: List[List[int]]) -> int:
        def dfs(i, j, k):
            grid[i][j] = 0                       # đã thăm
            path.append(str(k))                  # vào ô theo hướng k
            dirs = (-1, 0, 1, 0, -1)
            for h in range(1, 5):                # thứ tự thử hướng cố định
                x, y = i + dirs[h - 1], j + dirs[h]
                if 0 <= x < m and 0 <= y < n and grid[x][y]:
                    dfs(x, y, h)
            path.append(str(-k))                 # QUAY LUI

        paths, path = set(), []
        m, n = len(grid), len(grid[0])
        for i in range(m):
            for j in range(n):
                if grid[i][j]:
                    dfs(i, j, 0)
                    paths.add("".join(path))
                    path.clear()
        return len(paths)
```

**Template thay thế (chuẩn hóa tọa độ):** BFS/DFS lấy danh sách ô → trừ `min hàng`, `min cột` → `frozenset` → `set`.

**Checklist trước khi submit:**

- [ ] Liên thông theo **4 hướng** (không chéo)?
- [ ] Dấu vân tay **bất biến theo tịnh tiến** (trừ mốc hoặc chuỗi đường đi bắt đầu từ ô đầu)?
- [ ] Chuỗi đường đi có **ký hiệu quay lui** và thứ tự hướng **cố định**?
- [ ] Phần tử đưa vào `set` là **hashable** (chuỗi, tuple, frozenset)?
- [ ] Xoay/lật **không** được coi là giống nhau (đó là bài 711)?
- [ ] Đã xử lý lưới sửa tại chỗ (copy khi test), và độ sâu đệ quy tối đa `m * n`?
- [ ] Đã thử `[[0]]`, `[[1]]`, toàn đất, bàn cờ, hai ô chéo nhau?
- [ ] Time `O(m * n)`, Space `O(m * n)`?

**Ghi nhớ một câu:** *"Mỗi đảo cần một dấu vân tay không phụ thuộc vị trí: chuỗi đường đi DFS có quay lui, hoặc tọa độ trừ mốc; rồi bỏ vào set."*